# Distributed Alignment Search (DAS): An End-to-End Tutorial

**CS221M final project — Phase 1 (replication of course material).**

This notebook is a guided, runnable replication of the DAS causal-abstraction
pipeline (Geiger et al., 2023), built from the course notebooks. It has two
working centerpieces, in the order we recommend reading them:

1. **MLP DAS core** — the canonical *hierarchical-equality* task, where DAS
   learns an orthonormal rotation that cleanly partitions an MLP's hidden layer
   into the two intermediate variables `E1` and `E2`. (Source: lecture
   `10_causal_abstraction_ii`.)
2. **LM patching + DAS** — activation patching as a coarse layer x token-position
   baseline, then DAS to localize a factual attribute into a low-dimensional
   subspace of a real language model. (Source: lecture `08_causal_mediation_analysis`.)

**Main metric throughout: Interchange Intervention Accuracy (IIA)** — after we
swap an internal subspace from a *source* run into a *base* run, does the model
produce the counterfactual output predicted by the high-level causal model?

> Run on a **GPU runtime** (Colab: Runtime -> Change runtime type -> GPU).


## Part 0 — Setup

We install `pyvene` (the canonical DAS library from the paper authors; used for
the MLP section) plus the standard HF stack (used for the LM section). The LM
section uses plain forward hooks rather than a heavy framework, so the two parts
stay independent and debuggable.

In [ ]:
# Colab setup. Safe to re-run.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    %pip install -q pyvene==0.1.8 "transformers>=4.45" datasets matplotlib

# Optional: to use the (gated) Llama-3.2-1B in Part 3, authenticate once and accept
# the license at https://huggingface.co/meta-llama/Llama-3.2-1B . Otherwise the model
# loader falls back to a non-gated model automatically.
#   from huggingface_hub import login; login()  # paste an HF token

In [ ]:
import math
import random
import itertools

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else (
    "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE == "cpu":
    print("WARNING: no GPU detected. The MLP section will be slow; reduce N_TRAIN / N_CF below.")

## Part 1 — The idea in one page

**High-level causal model.** A causal model $\mathcal{M}$ is a set of variables
with mechanisms (functions of their parents). Our running example is the
*hierarchical equality* circuit:

- Inputs $A_1, A_2, A_3, A_4 \in \{0,1\}$.
- $E_1 = \mathrm{eq}(A_1, A_2)$, $E_2 = \mathrm{eq}(A_3, A_4)$.
- Output $C = \mathrm{eq}(E_1, E_2)$.

$E_1$ and $E_2$ are the *intermediate variables* we want to find inside a neural
network.

**Interchange intervention.** Take a `base` input and a `source` input. Run both.
Now overwrite the network's representation of (say) $E_1$ with the value it took
on the `source`, and finish the `base` forward pass. If the network truly
implements $\mathcal{M}$, the output should change to exactly what
$\mathcal{M}$ predicts when you set $E_1$ to the source value.

**IIA (Interchange Intervention Accuracy).** The fraction of (base, source) pairs
on which the post-intervention output matches the counterfactual label predicted
by $\mathcal{M}$. IIA = 1.0 means the chosen representation *is* the causal
variable.

**Activation patching** is the coarse version: overwrite the *entire* residual
stream at one (layer, token) location and measure IIA. It localizes information
to a layer/position but not to a *direction*.

**DAS** is the refined version: learn an orthonormal rotation $R$ of the hidden
state and patch only the first $k$ rotated coordinates. The learned subspace can
encode a variable while being orthogonal to others. As we will see in Part 2, $R$
plays the role of a **translation** $\tau$ between the network's coordinates and
the causal model's variables.

## Part 2 — MLP DAS core (centerpiece)

We train a small MLP to solve hierarchical equality, then use DAS to show its
hidden layer factorizes into an `E1` subspace and an `E2` subspace.

### 2.1 The high-level causal model

We implement $\mathcal{M}$ directly in a few lines of Python (mirroring the
`causalab` `CausalModel` used in lecture `10_causal_abstraction_ii`). A *trace* is
just a dict holding every variable's value for one input.

In [ ]:
def eq(a, b):
    "Equality on binary values: 1 if equal else 0."
    return int(a == b)

def sample_trace():
    "Sample one full run of the hierarchical-equality model M."
    a = {f"A{i}": random.randint(0, 1) for i in range(1, 5)}
    E1 = eq(a["A1"], a["A2"])
    E2 = eq(a["A3"], a["A4"])
    C = eq(E1, E2)
    return {**a, "E1": E1, "E2": E2, "C": C}

# sanity check: M on input (1,1,0,0) -> E1=1, E2=1, C=1
t = {"A1": 1, "A2": 1, "A3": 0, "A4": 0}
E1 = eq(t["A1"], t["A2"]); E2 = eq(t["A3"], t["A4"])
print("Input (1,1,0,0):  E1 =", E1, " E2 =", E2, " C =", eq(E1, E2))

### 2.2 Encoding inputs for the MLP

Each example uses **four fresh random vectors**. Positions in the same pair get
the *same* vector when their bits are equal and *different* vectors otherwise;
the two pairs always use distinct vectors. So the MLP can only detect equality by
comparing positions *within* a pair — exactly matching the causal structure of
$\mathcal{M}$. The 4 vectors of dim 4 concatenate into a 16-dim input.

In [ ]:
EMB_DIM = 4  # per-entity embedding dim; MLP input dim = EMB_DIM * 4 = 16

def randvec(n=EMB_DIM, lo=-1, hi=1):
    return np.array([round(random.uniform(lo, hi), 2) for _ in range(n)])

def make_input_tensor(trace):
    "Map an M trace to a 16-dim MLP input with fresh random entity vectors."
    e = [randvec() for _ in range(4)]
    vecs = [None] * 4
    vecs[0] = e[0]
    vecs[1] = e[0] if trace["A1"] == trace["A2"] else e[1]
    vecs[2] = e[2]
    vecs[3] = e[2] if trace["A3"] == trace["A4"] else e[3]
    return torch.tensor(np.concatenate(vecs), dtype=torch.float32)

print("MLP input dim:", EMB_DIM * 4)

### 2.3 Train the MLP

We use pyvene's `create_mlp_classifier` so the trained model plugs directly into
`IntervenableModel` later. Scale `N_TRAIN` down if you are on CPU.

In [ ]:
from pyvene.models.mlp.modelings_mlp import MLPConfig
from pyvene import create_mlp_classifier

H_DIM = 16
N_TRAIN = 262144      # lecture uses 1048576; smaller is plenty here
BATCH = 1024
N_EPOCHS = 3

def generate_factual_data(n):
    X, y = [], []
    for _ in range(n):
        tr = sample_trace()
        X.append(make_input_tensor(tr))
        y.append(tr["C"])
    return torch.stack(X), torch.tensor(y, dtype=torch.long)

X_train, y_train = generate_factual_data(N_TRAIN)
print("Train:", X_train.shape, " positives:", (y_train == 1).sum().item())

mlp_config = MLPConfig(h_dim=H_DIM, n_layer=3, pdrop=0.0, include_bias=True,
                       num_classes=2, squeeze_output=False, input_dim=EMB_DIM * 4)
_, _, mlp = create_mlp_classifier(mlp_config)

# Kaiming init for clean gradient flow through ReLU.
for name, p in mlp.named_parameters():
    if "weight" in name and p.dim() >= 2:
        torch.nn.init.kaiming_normal_(p, nonlinearity="relu")
    elif "bias" in name:
        torch.nn.init.zeros_(p)

mlp.to(DEVICE)
opt = torch.optim.Adam(mlp.parameters(), lr=0.01)

for epoch in range(N_EPOCHS):
    correct = total = 0
    perm = torch.randperm(X_train.shape[0])
    for s in range(0, X_train.shape[0], BATCH):
        idx = perm[s:s + BATCH]
        xb, yb = X_train[idx].to(DEVICE), y_train[idx].to(DEVICE)
        loss, logits = mlp(inputs_embeds=xb, labels=yb)
        opt.zero_grad(); loss.backward(); opt.step()
        correct += (logits.argmax(1) == yb).sum().item(); total += yb.shape[0]
    print(f"epoch {epoch+1}/{N_EPOCHS}  train acc {correct/total:.4f}")

In [ ]:
# Held-out factual accuracy
X_test, y_test = generate_factual_data(10000)
X_test, y_test = X_test.to(DEVICE), y_test.to(DEVICE)
mlp.eval()
with torch.no_grad():
    acc = (mlp(inputs_embeds=X_test)[0].argmax(1) == y_test).float().mean().item()
print(f"Held-out factual accuracy: {acc:.4f}")
print("The MLP solves the task -- but we don't yet know HOW its hidden layer encodes E1, E2.")

### 2.4 Counterfactual data for DAS

For each example we sample a `base` and two `source` inputs and compute the
counterfactual label after interchanging `E1` (from source 1) and/or `E2` (from
source 2). `intervention_id`: `0` = swap E1 only, `1` = swap E2 only, `2` = both.

In [ ]:
def generate_counterfactual_data(n, intervention_id=2):
    data = []
    for _ in range(n):
        base = sample_trace(); s1 = sample_trace(); s2 = sample_trace()
        cf_E1 = s1["E1"] if intervention_id in (0, 2) else base["E1"]
        cf_E2 = s2["E2"] if intervention_id in (1, 2) else base["E2"]
        data.append({
            "input_ids": make_input_tensor(base),
            "source_input_ids": torch.stack([make_input_tensor(s1), make_input_tensor(s2)]),
            "labels": torch.tensor(eq(cf_E1, cf_E2), dtype=torch.long),
            "intervention_id": intervention_id,
        })
    return data

N_CF = 256000
CF_BATCH = 6400
train_cf = generate_counterfactual_data(N_CF, intervention_id=2)
print("Counterfactual train examples:", len(train_cf), " keys:", list(train_cf[0].keys()))

### 2.5 The DAS model

We wrap the MLP in an `IntervenableModel` with **two linked** rotated-subspace
interventions on the hidden layer (`block_output`). Both share one rotation `R`
(`intervention_link_key=0`). In the rotated space we reserve dims `0:8` for `E1`
and `8:16` for `E2`.

In [ ]:
from torch.utils.data import DataLoader
from pyvene import (IntervenableModel, RotatedSpaceIntervention,
                    RepresentationConfig, IntervenableConfig)

def create_das_model(mlp, layer):
    config = IntervenableConfig(
        model_type=type(mlp),
        representations=[
            RepresentationConfig(layer, "block_output", "pos", 1,
                                 subspace_partition=None, intervention_link_key=0),
            RepresentationConfig(layer, "block_output", "pos", 1,
                                 subspace_partition=None, intervention_link_key=0),
        ],
        intervention_types=RotatedSpaceIntervention,
    )
    return IntervenableModel(config, mlp, use_fast=True)

def run_das_forward(das_model, batch, bs, emb=EMB_DIM):
    "Dispatch the interchange by intervention_id. E1 -> dims 0:8, E2 -> dims 8:16."
    iid = batch["intervention_id"][0].item()
    if iid == 2:
        _, cf = das_model(
            {"inputs_embeds": batch["input_ids"]},
            [{"inputs_embeds": batch["source_input_ids"][:, 0]},
             {"inputs_embeds": batch["source_input_ids"][:, 1]}],
            {"sources->base": ([[[0]] * bs, [[0]] * bs], [[[0]] * bs, [[0]] * bs])},
            subspaces=[[[i for i in range(0, emb * 2)]] * bs,
                       [[i for i in range(emb * 2, emb * 4)]] * bs])
    elif iid == 0:
        _, cf = das_model(
            {"inputs_embeds": batch["input_ids"]},
            [{"inputs_embeds": batch["source_input_ids"][:, 0]}, None],
            {"sources->base": ([[[0]] * bs, None], [[[0]] * bs, None])},
            subspaces=[[[i for i in range(0, emb * 2)]] * bs, None])
    elif iid == 1:
        _, cf = das_model(
            {"inputs_embeds": batch["input_ids"]},
            [None, {"inputs_embeds": batch["source_input_ids"][:, 1]}],
            {"sources->base": ([None, [[0]] * bs], [None, [[0]] * bs])},
            subspaces=[None, [[i for i in range(emb * 2, emb * 4)]] * bs])
    return cf

def batched_random_sampler(data, batch_size):
    order = list(range(len(data) // batch_size))
    random.shuffle(order)
    for b in order:
        for i in range(b * batch_size, (b + 1) * batch_size):
            yield i

print("DAS helpers ready.")

### 2.6 Train the rotation R

Only `R` is trainable; the MLP is frozen. We orthonormalize via pyvene's rotated
intervention and clip gradients, exactly as in the lecture.

In [ ]:
das_model = create_das_model(mlp, layer=0)
das_model.set_device(DEVICE)
das_model.disable_model_gradients()

opt_params = []
for k, v in das_model.interventions.items():
    opt_params += [{"params": v.rotate_layer.parameters()}]
    break  # linked: one shared rotation
das_opt = torch.optim.Adam(opt_params, lr=0.001)
print("Trainable params:", das_model.count_parameters())

DAS_EPOCHS = 10
for epoch in range(DAS_EPOCHS):
    das_model.model.train()
    ep_loss = ep_correct = ep_total = 0
    loader = DataLoader(train_cf, batch_size=CF_BATCH,
                        sampler=batched_random_sampler(train_cf, CF_BATCH))
    for batch in loader:
        batch["input_ids"] = batch["input_ids"].unsqueeze(1)
        batch["source_input_ids"] = batch["source_input_ids"].unsqueeze(2)
        bs = batch["input_ids"].shape[0]
        for k, v in batch.items():
            if isinstance(v, torch.Tensor):
                batch[k] = v.to(DEVICE)
        cf = run_das_forward(das_model, batch, bs)
        logits = cf[0].squeeze(1)
        labels = batch["labels"].squeeze().long()
        loss = torch.nn.CrossEntropyLoss()(logits, labels)
        das_opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(das_model.parameters(), 1.0)
        das_opt.step(); das_model.set_zero_grad()
        ep_loss += loss.item()
        ep_correct += (logits.argmax(1) == labels).sum().item(); ep_total += bs
    print(f"epoch {epoch+1}/{DAS_EPOCHS}  loss {ep_loss/(ep_total/CF_BATCH):.4f}  IIA {ep_correct/ep_total:.4f}")

### 2.7 Evaluate IIA (overall and per-variable)

High per-variable IIA confirms the rotation cleanly partitions the hidden layer:
dims `0:8` carry `E1`, dims `8:16` carry `E2`.

In [ ]:
def eval_iia(das_model, intervention_id, n=12800, batch_size=CF_BATCH):
    ds = generate_counterfactual_data(n, intervention_id=intervention_id)
    correct = total = 0
    with torch.no_grad():
        loader = DataLoader(ds, batch_size=batch_size,
                            sampler=batched_random_sampler(ds, batch_size))
        for batch in loader:
            batch["input_ids"] = batch["input_ids"].unsqueeze(1)
            batch["source_input_ids"] = batch["source_input_ids"].unsqueeze(2)
            bs = batch["input_ids"].shape[0]
            for k, v in batch.items():
                if isinstance(v, torch.Tensor):
                    batch[k] = v.to(DEVICE)
            cf = run_das_forward(das_model, batch, bs)
            logits = cf[0].squeeze(1)
            labels = batch["labels"].squeeze().long()
            correct += (logits.argmax(1) == labels).sum().item(); total += bs
    return correct / total

das_model.model.eval()
iia_e1 = eval_iia(das_model, 0)
iia_e2 = eval_iia(das_model, 1)
iia_both = eval_iia(das_model, 2)
print(f"Per-variable IIA:\n  E1 only:      {iia_e1:.4f}\n  E2 only:      {iia_e2:.4f}\n  Both E1 & E2: {iia_both:.4f}")

### 2.8 Ablation A — DAS vs. an untrained (random) rotation

A fresh `IntervenableModel` has a random orthonormal rotation. Its IIA is the
"does any random 8-dim subspace already encode E1/E2?" baseline. The gap to the
trained rotation is what DAS bought us.

In [ ]:
rand_das = create_das_model(mlp, layer=0)
rand_das.set_device(DEVICE)
rand_das.disable_model_gradients()
rand_das.model.eval()
rand_both = eval_iia(rand_das, 2)
rand_e1 = eval_iia(rand_das, 0)
rand_e2 = eval_iia(rand_das, 1)
print(f"Random rotation IIA -> E1: {rand_e1:.4f}  E2: {rand_e2:.4f}  both: {rand_both:.4f}")
print(f"Trained rotation IIA -> E1: {iia_e1:.4f}  E2: {iia_e2:.4f}  both: {iia_both:.4f}")

fig, ax = plt.subplots(figsize=(6, 4))
labels = ["E1 only", "E2 only", "Both"]
x = np.arange(len(labels)); w = 0.35
ax.bar(x - w/2, [rand_e1, rand_e2, rand_both], w, label="Random rotation")
ax.bar(x + w/2, [iia_e1, iia_e2, iia_both], w, label="Trained DAS")
ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylabel("IIA"); ax.set_ylim(0, 1.05)
ax.axhline(0.5, ls="--", c="gray", alpha=0.6, label="chance")
ax.set_title("DAS localizes E1/E2; a random subspace does not")
ax.legend(); plt.tight_layout(); plt.show()

### 2.9 Ablation B — subspace-dimension sweep

How many rotated dimensions does each variable's subspace need? We retrain DAS with
the `both` intervention but vary the size `d` of each variable's subspace (dims
`0:d` and `d:2d`).

**Diagnostic only.** To keep this cheap, each `d` is trained for just a few epochs
(`SWEEP_EPOCHS`), so the absolute IIA sits well below the fully-trained result in
§2.6. Read the curve for its *trend* (more dims → higher ceiling), not as a
benchmark of exact IIA. For the poster, prefer the trained-vs-random bar plot
(§2.8) as the headline MLP result.

In [ ]:
def create_das_model_partition(mlp, layer, d):
    "Two linked rotated interventions; each variable owns d dims."
    config = IntervenableConfig(
        model_type=type(mlp),
        representations=[
            RepresentationConfig(layer, "block_output", "pos", 1,
                                 subspace_partition=None, intervention_link_key=0),
            RepresentationConfig(layer, "block_output", "pos", 1,
                                 subspace_partition=None, intervention_link_key=0),
        ],
        intervention_types=RotatedSpaceIntervention,
    )
    return IntervenableModel(config, mlp, use_fast=True)

def run_forward_dims(dm, batch, bs, d):
    _, cf = dm(
        {"inputs_embeds": batch["input_ids"]},
        [{"inputs_embeds": batch["source_input_ids"][:, 0]},
         {"inputs_embeds": batch["source_input_ids"][:, 1]}],
        {"sources->base": ([[[0]] * bs, [[0]] * bs], [[[0]] * bs, [[0]] * bs])},
        subspaces=[[[i for i in range(0, d)]] * bs,
                   [[i for i in range(d, 2 * d)]] * bs])
    return cf

def train_eval_dim(d, epochs, n_cf=64000, n_eval=12800):
    dm = create_das_model_partition(mlp, 0, d)
    dm.set_device(DEVICE); dm.disable_model_gradients()
    params = []
    for k, v in dm.interventions.items():
        params += [{"params": v.rotate_layer.parameters()}]; break
    opt = torch.optim.Adam(params, lr=0.001)
    tr = generate_counterfactual_data(n_cf, 2)
    for _ in range(epochs):
        dm.model.train()
        loader = DataLoader(tr, batch_size=CF_BATCH,
                            sampler=batched_random_sampler(tr, CF_BATCH))
        for batch in loader:
            batch["input_ids"] = batch["input_ids"].unsqueeze(1)
            batch["source_input_ids"] = batch["source_input_ids"].unsqueeze(2)
            bs = batch["input_ids"].shape[0]
            for k, v in batch.items():
                if isinstance(v, torch.Tensor):
                    batch[k] = v.to(DEVICE)
            cf = run_forward_dims(dm, batch, bs, d)
            logits = cf[0].squeeze(1); labels = batch["labels"].squeeze().long()
            loss = torch.nn.CrossEntropyLoss()(logits, labels)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(dm.parameters(), 1.0)
            opt.step(); dm.set_zero_grad()
    # eval
    dm.model.eval(); ev = generate_counterfactual_data(n_eval, 2)
    correct = total = 0
    with torch.no_grad():
        loader = DataLoader(ev, batch_size=CF_BATCH,
                            sampler=batched_random_sampler(ev, CF_BATCH))
        for batch in loader:
            batch["input_ids"] = batch["input_ids"].unsqueeze(1)
            batch["source_input_ids"] = batch["source_input_ids"].unsqueeze(2)
            bs = batch["input_ids"].shape[0]
            for k, v in batch.items():
                if isinstance(v, torch.Tensor):
                    batch[k] = v.to(DEVICE)
            cf = run_forward_dims(dm, batch, bs, d)
            logits = cf[0].squeeze(1); labels = batch["labels"].squeeze().long()
            correct += (logits.argmax(1) == labels).sum().item(); total += bs
    return correct / total

# NOTE: each d is trained for only SWEEP_EPOCHS epochs with a fresh model, so
# absolute IIA values are lower than the main DAS result. Use this plot to see
# the *trend* (more dims = higher ceiling), not to benchmark exact IIA.
SWEEP_EPOCHS = 6
dims = [1, 2, 4, 8]
sweep = [(d, train_eval_dim(d, SWEEP_EPOCHS)) for d in dims]
for d, a in sweep:
    print(f"subspace dim per variable = {d:2d}  ->  IIA {a:.4f}")

ds_, accs_ = zip(*sweep)
plt.figure(figsize=(6, 4))
plt.plot(ds_, accs_, "o-")
plt.xlabel("Subspace dim per variable"); plt.ylabel("IIA (both)")
plt.title(f"Dim sweep ({SWEEP_EPOCHS} epochs each — undertrained diagnostic)")
plt.ylim(0, 1.05); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

### 2.10 Connecting back: R is a learned translation

The rotation $R$ DAS learned plays the role of the translation $\tau$ between the
MLP's coordinates and the causal model's variables. The first 8 rotated
coordinates *are* the network's representation of $E_1$; the next 8 are $E_2$.
High IIA means the constructive-abstraction equation holds: intervening on the
subspace is equivalent to intervening on the high-level variable.

## Part 3 — From toy MLP to a real language model

Now we repeat the logic on a pretrained LM. The high-level variable is the
**country attribute** of a city. Base prompt: *"Paris is in the country of"* ->
`France`. Source prompt: *"Tokyo is in the country of"* -> `Japan`. If we patch
the right place, the base output should flip to `Japan`.

We use plain forward hooks (no extra framework) so the intervention is fully
transparent. Model: `HuggingFaceTB/SmolLM2-360M` (small, knows these facts).

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# ── Model selection ──────────────────────────────────────────────────────────
# Llama-3.2-1B is the DEFAULT (strongest factual recall; matches the nnsight
# tutorials). It is GATED on HuggingFace, so to actually get it you must, ONCE:
#     from huggingface_hub import login; login()     # paste an HF token
#   and accept the license at https://huggingface.co/meta-llama/Llama-3.2-1B
# If Llama can't load (no token / license not accepted / offline), we fall back to
# the next model so the notebook still runs end-to-end — but we print a LOUD
# warning so you know you're not on the preferred model.
PREFERRED_MODEL = "meta-llama/Llama-3.2-1B"
FALLBACK_MODELS = ["Qwen/Qwen2.5-0.5B", "gpt2-large", "gpt2"]
MODEL_CANDIDATES = [PREFERRED_MODEL] + FALLBACK_MODELS

# fp32 everywhere keeps the hand-rolled rotation R (float32) dtype-consistent and
# avoids fp16 precision issues during DAS backprop. A 1B model in fp32 fits a T4.
LOAD_DTYPE = torch.float32

def try_load(name):
    tk = AutoTokenizer.from_pretrained(name)
    if tk.pad_token is None:
        tk.pad_token = tk.eos_token
    m = AutoModelForCausalLM.from_pretrained(name, torch_dtype=LOAD_DTYPE).eval().to(DEVICE)
    return tk, m

tok = lm = LM_NAME = None
for cand in MODEL_CANDIDATES:
    try:
        tok, lm = try_load(cand)
        LM_NAME = cand
        print(f"Loaded: {cand}")
        break
    except Exception as e:
        tag = "PREFERRED" if cand == PREFERRED_MODEL else "fallback"
        hint = " (gated: login + accept license)" if "llama" in cand.lower() else ""
        print(f"  [{tag}] could not load {cand}{hint}: {type(e).__name__}: {str(e)[:120]}")
if lm is None:
    raise RuntimeError("No model could be loaded from MODEL_CANDIDATES.")

# Tell the user, loudly, if we are NOT on the preferred model.
if LM_NAME != PREFERRED_MODEL:
    bar = "!" * 74
    print("\n" + bar)
    print(f"!!  FELL BACK to '{LM_NAME}'.")
    print(f"!!  The default model '{PREFERRED_MODEL}' did NOT load (usually the HF gate).")
    print( "!!  To use Llama: run  `from huggingface_hub import login; login()`,")
    print(f"!!  accept the license at https://huggingface.co/{PREFERRED_MODEL},")
    print( "!!  then re-run this cell. Results below are valid but on a smaller model.")
    print(bar + "\n")
else:
    print(f"\n>>> Using preferred model: {LM_NAME}\n")

# Freeze the model: we only ever train the rotation R. Gradients still FLOW through
# the frozen layers back to R; freezing just avoids storing grads on model weights.
lm.requires_grad_(False)

def get_layers(model):
    for attr in ["model.layers", "transformer.h", "model.decoder.layers"]:
        obj = model
        try:
            for p in attr.split("."):
                obj = getattr(obj, p)
            return list(obj)
        except AttributeError:
            continue
    raise ValueError("cannot find layers")

def hidden_size(model):
    for a in ["hidden_size", "n_embd", "d_model"]:
        if hasattr(model.config, a):
            return int(getattr(model.config, a))
    raise ValueError("no hidden size")

N_LAYERS = len(get_layers(lm))
HID = hidden_size(lm)
print(f"{LM_NAME}: {N_LAYERS} layers, hidden {HID}")

# Tokenize consistently EVERYWHERE (training, caching, length, labels) so token
# positions line up. Some tokenizers (Llama, Qwen) prepend a BOS token; as long as
# every code path uses the same call, positions stay aligned (BOS just becomes pos 0).
def enc(prompt):
    return tok(prompt, return_tensors="pt").to(DEVICE)

def prompt_len(prompt):
    return len(tok(prompt)["input_ids"])

def value_token_id(value):
    # the answer token itself must NOT carry a BOS, hence add_special_tokens=False
    return tok.encode(" " + value.strip(), add_special_tokens=False)[0]

### 3.1 Build a clean, model-correct dataset

We keep only cities the model already answers correctly, and pair each city with
a source city of a *different* country. We also fix all prompts to the same token
length so a single token position aligns across examples (we use cities that are
single tokens for clean patching).

In [ ]:
# A broad list of capital -> country facts. We keep only the ones the model actually
# predicts correctly, then restrict to a single prompt length so token positions line up.
CITIES = [
    ("Paris", "France"), ("Tokyo", "Japan"), ("Berlin", "Germany"),
    ("Cairo", "Egypt"), ("Madrid", "Spain"), ("Rome", "Italy"),
    ("Moscow", "Russia"), ("London", "England"), ("Athens", "Greece"),
    ("Dublin", "Ireland"), ("Lisbon", "Portugal"), ("Vienna", "Austria"),
    ("Oslo", "Norway"), ("Warsaw", "Poland"), ("Bangkok", "Thailand"),
    ("Amsterdam", "Netherlands"), ("Brussels", "Belgium"), ("Stockholm", "Sweden"),
    ("Helsinki", "Finland"), ("Budapest", "Hungary"), ("Prague", "Czechia"),
    ("Beijing", "China"), ("Seoul", "Korea"), ("Ottawa", "Canada"),
    ("Canberra", "Australia"), ("Havana", "Cuba"), ("Baghdad", "Iraq"),
    ("Tehran", "Iran"), ("Ankara", "Turkey"), ("Nairobi", "Kenya"),
]
PROMPT = "{city} is in the country of"

def next_token_id(prompt):
    with torch.no_grad():
        return lm(**enc(prompt)).logits[0, -1].argmax().item()

# keep cities the model gets right AND whose prompt tokenizes to a fixed length
correct_all = []
for city, country in CITIES:
    p = PROMPT.format(city=city)
    L = prompt_len(p)
    if next_token_id(p) == value_token_id(country):
        correct_all.append((city, country, L))

from collections import Counter
print("Model-correct cities:", [(c, k) for c, k, _ in correct_all])
target_len = Counter([L for *_, L in correct_all]).most_common(1)[0][0]
correct = [(c, k) for c, k, L in correct_all if L == target_len]
N_POS = target_len
print(f"Using {len(correct)} cities at prompt length {N_POS} tokens "
      f"-> up to {len(correct)*(len(correct)-1)} ordered pairs.")

In [ ]:
# Full cross-product pairing: every model-correct city is a base against every
# other city as source. Cause label = source country (output should flip);
# isolate label = base country (output should stay if we DON'T touch the subspace).
import random
random.seed(SEED if "SEED" in globals() else 0)

def build_cross_product_pairs(items):
    pairs = []
    for i, (bc, bk) in enumerate(items):
        for j, (sc, sk) in enumerate(items):
            if i == j or sk == bk:
                continue  # need a genuinely different target country
            pairs.append({
                "base_prompt": PROMPT.format(city=bc),
                "source_prompt": PROMPT.format(city=sc),
                "cause_label": value_token_id(sk),
                "isolate_label": value_token_id(bk),
            })
    return pairs

all_pairs = build_cross_product_pairs(correct)
random.shuffle(all_pairs)
n_eval = max(1, int(round(0.2 * len(all_pairs))))
eval_pairs = all_pairs[:n_eval]
train_pairs = all_pairs[n_eval:]
print(f"{len(all_pairs)} total pairs -> {len(train_pairs)} train / {len(eval_pairs)} held-out eval (~20%)")
print("Example pair:", {k: all_pairs[0][k] for k in ["base_prompt", "source_prompt"]})

### 3.2 Forward hooks: collect activations and patch

`collect_layer_outputs` caches every layer's residual stream for a prompt.
`forward_with_patch` re-runs the base prompt while overwriting the hidden state at
one `(layer, pos)` with a (possibly transformed) source vector.

In [ ]:
def collect_layer_outputs(prompt):
    layers = get_layers(lm)
    out = [None] * len(layers)
    handles = []
    def mk(li):
        def hook(m, inp, o):
            hs = o[0] if isinstance(o, tuple) else o
            out[li] = hs[0].detach()
        return hook
    for li, layer in enumerate(layers):
        handles.append(layer.register_forward_hook(mk(li)))
    ids = tok(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        lm(**ids)
    for h in handles:
        h.remove()
    return out  # list[layer] -> [seq, hidden]

def forward_with_patch(base_prompt, source_h, layer_idx, pos, transform_fn=None, requires_grad=False):
    layers = get_layers(lm)
    def hook(m, inp, o):
        hs = o[0] if isinstance(o, tuple) else o
        h_base = hs[0, pos, :]
        h_new = transform_fn(h_base, source_h) if transform_fn is not None else source_h
        hs2 = hs.clone(); hs2[0, pos, :] = h_new
        return (hs2,) + o[1:] if isinstance(o, tuple) else hs2
    handle = layers[layer_idx].register_forward_hook(hook)
    ids = tok(base_prompt, return_tensors="pt").to(DEVICE)
    ctx = torch.enable_grad() if requires_grad else torch.no_grad()
    with ctx:
        out = lm(**ids)
    handle.remove()
    return out.logits[0, -1]

# Cache activations ONCE per unique source prompt (cross-product reuses each city
# many times as a source, so per-pair caching would be hugely redundant).
source_cache = {}
for ex in all_pairs:
    sp = ex["source_prompt"]
    if sp not in source_cache:
        source_cache[sp] = collect_layer_outputs(sp)  # list[layer] [seq,hidden]
print("Cached source activations for", len(source_cache), "unique prompts.")

### 3.3 Activation-patching baseline: layer x position IIA heatmap

For every `(layer, pos)` we patch the full residual vector and measure **cause
IIA** — did the output flip to the *source* country? The bright region tells us
*where* the country attribute lives.

In [ ]:
# Sweep over (layer, position) on a subset of the TRAIN pairs for speed.
sweep_pairs = train_pairs[:40]
cause_grid = np.zeros((N_LAYERS, N_POS))
for li in range(N_LAYERS):
    for pos in range(N_POS):
        c = 0
        for ex in sweep_pairs:
            src_h = source_cache[ex["source_prompt"]][li][pos].to(DEVICE)
            logits = forward_with_patch(ex["base_prompt"], src_h, li, pos)
            if logits.argmax().item() == ex["cause_label"]:
                c += 1
        cause_grid[li, pos] = c / len(sweep_pairs)
    print(f"layer {li+1}/{N_LAYERS} done", end="\r")
print()

best_layer, best_pos = np.unravel_index(cause_grid.argmax(), cause_grid.shape)
print(f"Best full-patch cell: layer {best_layer}, pos {best_pos}, cause IIA {cause_grid[best_layer,best_pos]:.3f}")

tok_labels = tok.convert_ids_to_tokens(tok(sweep_pairs[0]["base_prompt"])["input_ids"])
plt.figure(figsize=(max(6, N_POS), max(5, N_LAYERS * 0.3)))
plt.imshow(cause_grid, aspect="auto", origin="lower", cmap="viridis", vmin=0, vmax=1)
plt.colorbar(label="Cause IIA")
plt.xticks(range(N_POS), tok_labels, rotation=45, ha="right")
plt.yticks(range(N_LAYERS))
plt.xlabel("Token position"); plt.ylabel("Layer")
plt.title("Activation patching: where does 'country' live?")
plt.tight_layout(); plt.show()

### 3.4 DAS on the LM: localize the attribute to a subspace

At the best `(layer, pos)` we learn an orthonormal rotation `R` and patch only
the first `k` rotated dimensions. We keep `R` orthonormal by QR-projecting after
each step (the same recipe as the canonical DAS rotation). We train to flip the
output to the source country (cause), then report cause IIA and **isolate IIA**
(did *other* runs keep their base answer? a subspace that is too broad will hurt
this).

In [ ]:
K = 32                       # subspace dim to patch
DAS_LAYER, DAS_POS = int(best_layer), int(best_pos)
# Bigger models cost more per backprop step; scale epochs down so the demo stays ~minutes.
LM_DAS_EPOCHS = 30 if HID <= 1280 else 15

def project_orthonormal(R):
    Q, _ = torch.linalg.qr(R)
    return Q

R = torch.eye(HID, device=DEVICE, requires_grad=True)
r_opt = torch.optim.Adam([R], lr=1e-3)

for epoch in range(LM_DAS_EPOCHS):
    random.shuffle(train_pairs)
    total_loss = 0.0
    for ex in train_pairs:
        src_h = source_cache[ex["source_prompt"]][DAS_LAYER][DAS_POS].to(DEVICE)
        R_orth = project_orthonormal(R)
        def transform_fn(h_base, h_src=src_h, Ro=R_orth):
            fb = h_base @ Ro; fs = h_src @ Ro
            fp = torch.cat([fs[:K], fb[K:]], dim=0)
            return fp @ Ro.T
        logits = forward_with_patch(ex["base_prompt"], src_h, DAS_LAYER, DAS_POS,
                                    transform_fn=transform_fn, requires_grad=True)
        loss = F.cross_entropy(logits.unsqueeze(0),
                               torch.tensor([ex["cause_label"]], device=DEVICE))
        r_opt.zero_grad(); loss.backward(); r_opt.step()
        with torch.no_grad():
            R.data = project_orthonormal(R)
        total_loss += loss.item()
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"epoch {epoch+1}/{LM_DAS_EPOCHS}  loss {total_loss/len(train_pairs):.4f}")

In [ ]:
# ── helpers used here AND in Phase 2 ────────────────────────────────────────
def full_patch_iia(eval_set, cache, layer, pos):
    """True held-out full-vector patch IIA (no rotation, no subspace)."""
    correct = 0
    for ex in eval_set:
        src_h = cache[ex["source_prompt"]][layer][pos].to(DEVICE)
        logits = forward_with_patch(ex["base_prompt"], src_h, layer, pos)
        correct += int(logits.argmax().item() == ex["cause_label"])
    return correct / len(eval_set)

# ── evaluate the learned rotation on the HELD-OUT pairs ──────────────────────
#   cause   IIA  — patch learned K-dim subspace from source  -> should flip to source country
#   isolate IIA  — patch the COMPLEMENT (HID-K dims) from source -> should stay base country
def das_eval(R, eval_set, mode="cause", k=K):
    R = project_orthonormal(R).detach()
    correct = 0
    for ex in eval_set:
        src_h = source_cache[ex["source_prompt"]][DAS_LAYER][DAS_POS].to(DEVICE)
        def transform_fn(h_base, h_src=src_h, Ro=R):
            fb = h_base @ Ro; fs = h_src @ Ro
            if mode == "cause":
                fp = torch.cat([fs[:k], fb[k:]], dim=0)
            else:
                fp = torch.cat([fb[:k], fs[k:]], dim=0)
            return fp @ Ro.T
        logits = forward_with_patch(ex["base_prompt"], src_h, DAS_LAYER, DAS_POS,
                                    transform_fn=transform_fn)
        label = ex["cause_label"] if mode == "cause" else ex["isolate_label"]
        correct += int(logits.argmax().item() == label)
    return correct / len(eval_set)

# ── primary results ───────────────────────────────────────────────────────────
full_cause_train = cause_grid[DAS_LAYER, DAS_POS]          # train-sweep reference
full_cause_eval  = full_patch_iia(eval_pairs, source_cache, DAS_LAYER, DAS_POS)
das_cause        = das_eval(R, eval_pairs, mode="cause")
das_isolate      = das_eval(R, eval_pairs, mode="isolate")

# ── controls: identity subspace + 3 random rotations ─────────────────────────
# These answer: "does DAS learn something, or does *any* 32-dim slice work?"
I = torch.eye(HID, device=DEVICE)
id_cause   = das_eval(I, eval_pairs, mode="cause")
id_isolate = das_eval(I, eval_pairs, mode="isolate")

rand_causes, rand_isolates = [], []
for _ in range(3):
    Rr = torch.randn(HID, HID, device=DEVICE)
    rand_causes.append(das_eval(Rr, eval_pairs, mode="cause"))
    rand_isolates.append(das_eval(Rr, eval_pairs, mode="isolate"))
rand_cause   = sum(rand_causes)   / len(rand_causes)
rand_isolate = sum(rand_isolates) / len(rand_isolates)

# ── print ─────────────────────────────────────────────────────────────────────
print(f"Held-out eval ({len(eval_pairs)} pairs) at layer {DAS_LAYER}, pos {DAS_POS}:")
print(f"  Full-vector patch  (train-sweep): {full_cause_train:.3f}  <- reference from sweep subset")
print(f"  Full-vector patch  (held-out):    {full_cause_eval:.3f}  <- true held-out baseline")
print(f"  DAS {K}-dim subspace  cause IIA:  {das_cause:.3f}")
print(f"  DAS complement       isolate IIA: {das_isolate:.3f}")
print(f"  Identity subspace    cause IIA:   {id_cause:.3f}  (control)")
print(f"  Random rotation (×3) cause IIA:   {rand_cause:.3f}  (control)")

# ── bar chart ─────────────────────────────────────────────────────────────────
labels  = ["full\npatch", f"DAS {K}d\ncause", f"DAS {K}d\nisolate",
           "identity\ncause", "random\ncause"]
values  = [full_cause_eval, das_cause, das_isolate, id_cause, rand_cause]
colors  = ["#888", "#1f77b4", "#2ca02c", "#ff7f0e", "#d62728"]
plt.figure(figsize=(7, 4))
bars = plt.bar(labels, values, color=colors)
plt.axhline(rand_cause, color="#d62728", linestyle="--", linewidth=0.8, label="random rotation")
plt.ylim(0, 1.1); plt.ylabel("IIA")
plt.title(f"Country DAS — held-out eval (layer {DAS_LAYER}, pos {DAS_POS}, K={K})")
for bar, v in zip(bars, values):
    plt.text(bar.get_x() + bar.get_width()/2, v + 0.02, f"{v:.2f}", ha="center", fontsize=9)
plt.tight_layout(); plt.show()

### 3.5 What we showed

- **Activation patching** localized the country attribute to a band of
  layers/positions (coarse: a whole residual vector).
- **DAS** then found a low-dimensional *direction* at the best location that carries
  the attribute — the same method scaling from the toy MLP to a real LM.

**Read this as a pipeline bridge, not a deep world-knowledge result.** Caveats to
keep on the poster:

- The model is whatever loaded in §3 — in our run **Qwen2.5-0.5B**, not Llama
  (the printout above states which one).
- We intervene on the **first next-token** of the country string, not full-string
  generation.
- The split is **held-out pairs**, not held-out *cities* — train and eval share the
  same city/country vocabulary.
- The best site is **very late** (e.g. layer ~22/24 at the final prompt token), so
  this is closer to *answer/logit-state* steering than a deep "country concept".

The identity- and random-subspace controls below are what license the claim that the
*learned* rotation matters, rather than any 32-dim slice working.

## Part 4 — Phase 2: Entity binding with DAS

Part 3 localized a *single global fact* (a city's country). Phase 2 asks a harder,
more interesting question: when a prompt contains **several entities each bound to
its own attribute**, where does the model store *"the attribute that belongs to the
queried entity"*?

Task (in-context, pure copying — no world knowledge needed):

```
Pete has the jam. Ann has the pie. Pete has the ___   -> jam
```

The answer is fully determined by the context, so a model with working induction
heads can solve it. The *causal variable* we target is the **binding** of the queried
entity to its value. The counterfactual swaps the values:

```
base:   Pete has the jam.  Ann has the pie.  Pete has the ___   (factual: jam)
source: Pete has the milk. Ann has the tea.  Pete has the ___   (factual: milk)
```

- **cause IIA** — patch the binding subspace from `source` -> output should flip to
  `milk` (the source's value for Pete).
- **isolate IIA** — patch the *complement* subspace from `source` -> output should
  stay `jam` (the binding lives in the subspace and nowhere else).

We reuse every tool built in Part 3 (`collect_layer_outputs`, `forward_with_patch`,
`project_orthonormal`), just on a new dataset.

In [ ]:
# Build the entity-binding dataset.
NAMES = ["Pete", "Ann", "Bob", "Sue", "Tom", "Joe", "Kate", "Mark", "Lucy", "Paul"]
ITEMS = ["jam", "pie", "cake", "milk", "ham", "tea", "egg", "bread", "soup", "rice"]
EB_TEMPLATE = "{e1} has the {a1}. {e2} has the {a2}. {e1} has the"

def eb_prompt(e1, a1, e2, a2):
    return EB_TEMPLATE.format(e1=e1, a1=a1, e2=e2, a2=a2)

rng = random.Random(SEED if "SEED" in globals() else 0)
cand, seen = [], set()
for _ in range(2000):
    e1, e2 = rng.sample(NAMES, 2)
    a1, a2 = rng.sample(ITEMS, 2)            # base assignment
    a1s, a2s = rng.sample(ITEMS, 2)          # source assignment
    if a1s == a1:                            # need the queried value to actually change
        continue
    bp = eb_prompt(e1, a1, e2, a2)
    sp = eb_prompt(e1, a1s, e2, a2s)
    if (bp, sp) in seen:
        continue
    seen.add((bp, sp))
    cand.append((e1, a1, e2, a2, a1s, a2s, bp, sp))

# Keep only pairs the model already binds correctly (base->a1, source->a1s) and that
# share one fixed prompt length so token positions align.
def n_tokens(p):
    return prompt_len(p)

ok = []
for (e1, a1, e2, a2, a1s, a2s, bp, sp) in cand:
    if n_tokens(bp) != n_tokens(sp):
        continue
    if next_token_id(bp) == value_token_id(a1) and next_token_id(sp) == value_token_id(a1s):
        ok.append((bp, sp, a1, a1s, n_tokens(bp)))

from collections import Counter
EB_POS_N = Counter([L for *_, L in ok]).most_common(1)[0][0]
ok = [(bp, sp, a1, a1s) for (bp, sp, a1, a1s, L) in ok if L == EB_POS_N]

eb_all = [{
    "base_prompt": bp, "source_prompt": sp,
    "cause_label": value_token_id(a1s),     # flip to source's value for the queried entity
    "isolate_label": value_token_id(a1),    # otherwise keep base value
} for (bp, sp, a1, a1s) in ok]
rng.shuffle(eb_all)

n_eval = max(1, int(round(0.2 * len(eb_all))))
eb_eval, eb_train = eb_all[:n_eval], eb_all[n_eval:]
print(f"Entity-binding: {len(eb_all)} model-correct pairs at length {EB_POS_N} "
      f"-> {len(eb_train)} train / {len(eb_eval)} held-out eval")
if eb_all:
    print("Example base:", eb_all[0]["base_prompt"], "| source:", eb_all[0]["source_prompt"])

# cache source activations once per unique source prompt
eb_cache = {}
for ex in eb_all:
    sp = ex["source_prompt"]
    if sp not in eb_cache:
        eb_cache[sp] = collect_layer_outputs(sp)
print("Cached", len(eb_cache), "unique source prompts.")

### 4.1 Where does the binding live? (patching sweep)

Same layer x position cause-IIA heatmap as Part 3, now over the entity-binding pairs.

In [ ]:
def patch_sweep(pairs_subset, cache, n_layers, n_pos):
    grid = np.zeros((n_layers, n_pos))
    for li in range(n_layers):
        for pos in range(n_pos):
            c = 0
            for ex in pairs_subset:
                src_h = cache[ex["source_prompt"]][li][pos].to(DEVICE)
                logits = forward_with_patch(ex["base_prompt"], src_h, li, pos)
                if logits.argmax().item() == ex["cause_label"]:
                    c += 1
            grid[li, pos] = c / len(pairs_subset)
        print(f"layer {li+1}/{n_layers} done", end="\r")
    print()
    return grid

eb_sweep_pairs = eb_train[:40]
eb_grid = patch_sweep(eb_sweep_pairs, eb_cache, N_LAYERS, EB_POS_N)
eb_best_layer, eb_best_pos = np.unravel_index(eb_grid.argmax(), eb_grid.shape)
print(f"Best full-patch cell: layer {eb_best_layer}, pos {eb_best_pos}, cause IIA {eb_grid[eb_best_layer,eb_best_pos]:.3f}")

eb_tok_labels = tok.convert_ids_to_tokens(tok(eb_sweep_pairs[0]["base_prompt"])["input_ids"])
plt.figure(figsize=(max(7, EB_POS_N), max(5, N_LAYERS * 0.3)))
plt.imshow(eb_grid, aspect="auto", origin="lower", cmap="magma", vmin=0, vmax=1)
plt.colorbar(label="Cause IIA")
plt.xticks(range(EB_POS_N), eb_tok_labels, rotation=45, ha="right")
plt.yticks(range(N_LAYERS))
plt.xlabel("Token position"); plt.ylabel("Layer")
plt.title("Entity binding: where does the queried entity's value live?")
plt.tight_layout(); plt.show()

### 4.2 Train a DAS rotation on the binding subspace

In [ ]:
def train_das(train, cache, layer, pos, hid, k, epochs, lr=1e-3, log_every=5):
    R = torch.eye(hid, device=DEVICE, requires_grad=True)
    opt = torch.optim.Adam([R], lr=lr)
    for epoch in range(epochs):
        rng.shuffle(train)
        total = 0.0
        for ex in train:
            src_h = cache[ex["source_prompt"]][layer][pos].to(DEVICE)
            Ro = project_orthonormal(R)
            def transform_fn(h_base, h_src=src_h, Rm=Ro):
                fb = h_base @ Rm; fs = h_src @ Rm
                return torch.cat([fs[:k], fb[k:]], dim=0) @ Rm.T
            logits = forward_with_patch(ex["base_prompt"], src_h, layer, pos,
                                        transform_fn=transform_fn, requires_grad=True)
            loss = F.cross_entropy(logits.unsqueeze(0),
                                   torch.tensor([ex["cause_label"]], device=DEVICE))
            opt.zero_grad(); loss.backward(); opt.step()
            with torch.no_grad():
                R.data = project_orthonormal(R)
            total += loss.item()
        if (epoch + 1) % log_every == 0 or epoch == 0:
            print(f"epoch {epoch+1}/{epochs}  loss {total/len(train):.4f}")
    return R

EB_K = 32
EB_LAYER, EB_POS = int(eb_best_layer), int(eb_best_pos)
EB_EPOCHS = 30 if HID <= 1280 else 15
R_eb = train_das(eb_train, eb_cache, EB_LAYER, EB_POS, HID, EB_K, EB_EPOCHS)

### 4.3 Evaluate on held-out pairs

In [ ]:
# ── primary results (held-out) ───────────────────────────────────────────────
eb_full_train = eb_grid[EB_LAYER, EB_POS]          # train-sweep reference
eb_full_eval  = full_patch_iia(eb_eval, eb_cache, EB_LAYER, EB_POS)   # true held-out
eb_cause      = das_eval_generic(R_eb, eb_eval, eb_cache, EB_LAYER, EB_POS, "cause",   EB_K)
eb_isolate    = das_eval_generic(R_eb, eb_eval, eb_cache, EB_LAYER, EB_POS, "isolate", EB_K)

# ── controls: identity subspace + 3 random rotations ─────────────────────────
I_eb = torch.eye(HID, device=DEVICE)
eb_id_cause   = das_eval_generic(I_eb, eb_eval, eb_cache, EB_LAYER, EB_POS, "cause",   EB_K)
eb_id_isolate = das_eval_generic(I_eb, eb_eval, eb_cache, EB_LAYER, EB_POS, "isolate", EB_K)

eb_rand_causes, eb_rand_isolates = [], []
for _ in range(3):
    Rr = torch.randn(HID, HID, device=DEVICE)
    eb_rand_causes.append(  das_eval_generic(Rr, eb_eval, eb_cache, EB_LAYER, EB_POS, "cause",   EB_K))
    eb_rand_isolates.append(das_eval_generic(Rr, eb_eval, eb_cache, EB_LAYER, EB_POS, "isolate", EB_K))
eb_rand_cause   = sum(eb_rand_causes)   / len(eb_rand_causes)
eb_rand_isolate = sum(eb_rand_isolates) / len(eb_rand_isolates)

# ── print ─────────────────────────────────────────────────────────────────────
print(f"Entity-binding held-out eval ({len(eb_eval)} pairs) at layer {EB_LAYER}, pos {EB_POS}:")
print(f"  Full-vector patch  (train-sweep): {eb_full_train:.3f}  <- reference from sweep subset")
print(f"  Full-vector patch  (held-out):    {eb_full_eval:.3f}  <- true held-out baseline")
print(f"  DAS {EB_K}-dim subspace  cause IIA:  {eb_cause:.3f}")
print(f"  DAS complement       isolate IIA: {eb_isolate:.3f}")
print(f"  Identity subspace    cause IIA:   {eb_id_cause:.3f}  (control)")
print(f"  Random rotation (×3) cause IIA:   {eb_rand_cause:.3f}  (control)")

# ── bar chart ─────────────────────────────────────────────────────────────────
labels  = ["full\npatch", f"DAS {EB_K}d\ncause", f"DAS {EB_K}d\nisolate",
           "identity\ncause", "random\ncause"]
values  = [eb_full_eval, eb_cause, eb_isolate, eb_id_cause, eb_rand_cause]
colors  = ["#888", "#1f77b4", "#2ca02c", "#ff7f0e", "#d62728"]
plt.figure(figsize=(7, 4))
bars = plt.bar(labels, values, color=colors)
plt.axhline(eb_rand_cause, color="#d62728", linestyle="--", linewidth=0.8, label="random rotation")
plt.ylim(0, 1.1); plt.ylabel("IIA")
plt.title(f"Entity-binding DAS — held-out eval (layer {EB_LAYER}, pos {EB_POS}, K={EB_K})")
for bar, v in zip(bars, values):
    plt.text(bar.get_x() + bar.get_width()/2, v + 0.02, f"{v:.2f}", ha="center", fontsize=9)
plt.tight_layout(); plt.show()

### 4.4 What Phase 2 showed, and what's next

A high `cause IIA` means the learned 32-dim subspace is usually **sufficient** to
rewrite the value bound to the queried entity. The `isolate IIA` (≈0.82 in our run)
is high but **not** 1.0 — patching the complement still flips the answer some of the
time. The honest reading is a **partially localized** binding subspace: the
entity→value binding is *largely* captured by a compact subspace, but not perfectly
separated from the rest of the representation.

Even so, this is a more interesting, *relational* claim than the single-fact result
in Part 3 — and the identity/random controls above show the learned rotation is
doing real work (DAS cause ≫ random/identity).

Remaining directions (out of scope here):

1. **Positional confound control** — re-run with the queried entity in second
   position to confirm DAS tracks the binding, not an absolute position.
2. **Held-out entities/items** — split on names/objects, not just on pairs.
3. **Concept DAS** (Bao et al., 2026) — push the learned-subspace idea to
   higher-level concepts rather than single tokens.